In [ ]:
# 0. Cấu hình
SEEDS = [0, 1, 2]
EPOCHS = 200
PATIENCE = 30
CLIP = 1.0
BATCH = 32
LR = 1e-3
WD = 0.0
LABEL_SMOOTH = 0.1
HEAD_DROPOUT = 0.5
STGCN_WIDTH = 120
RUN_FINAL_TEST = True
EVAL_SPLIT = "test"

In [ ]:
# 1. Môi trường
import os, glob, json, time, math, random, copy, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")
hits = sorted(glob.glob("/kaggle/input/**/data.npz", recursive=True)) or sorted(glob.glob("**/data.npz", recursive=True))
DATA_DIR = os.path.dirname(hits[0])
OUT = "/kaggle/working/results" if os.path.exists("/kaggle/working") else "results"
os.makedirs(OUT + "/runs", exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DATA_DIR, DEV, torch.__version__)

In [ ]:
# 2. Dữ liệu
Z = np.load(f"{DATA_DIR}/data.npz")
SIGNS = json.load(open(f"{DATA_DIR}/signs.json", encoding="utf-8"))
NAMES = [k for k, _ in sorted(SIGNS.items(), key=lambda kv: kv[1])]
N_CLS = len(NAMES)
meta = pd.read_csv(f"{DATA_DIR}/data.meta.csv")

def load(split):
    X = np.concatenate([Z["X_" + split], Z["M_" + split]], -1).astype(np.float32)
    return X, Z["y_" + split].astype(np.int64), Z["signer_" + split], Z["id_" + split]

Xtr, ytr, gtr, itr = load("train")
Xva, yva, gva, iva = load("val")
gte = set(Z["signer_test"])
assert not (set(gtr) & set(gva)) and not (set(gtr) & gte) and not (set(gva) & gte)
assert sorted(NAMES) == NAMES and list(SIGNS.values()) == list(range(N_CLS))
lo, hi = len(ytr), len(ytr) + len(yva)
assert (meta.sample_id.to_numpy()[:lo] == itr).all() and (meta.sample_id.to_numpy()[lo:hi] == iva).all()
sva = meta.status.to_numpy()[lo:hi]
print(f"train {Xtr.shape}  val {Xva.shape}  người ký train/val/test: {len(set(gtr))}/{len(set(gva))}/{len(gte)}")
print("val:", pd.Series(sva).value_counts().to_dict(), "| train:", pd.Series(meta.status.to_numpy()[:lo]).value_counts().to_dict())

In [ ]:
# 3. Chuẩn hóa
D = Xtr.shape[-1]
MEAN = Xtr.reshape(-1, D).mean(0)
STD = np.maximum(Xtr.reshape(-1, D).std(0), 1e-3)
MEAN[100:], STD[100:] = 0.0, 1.0
MEAN, STD = MEAN.astype(np.float32), STD.astype(np.float32)
Ttr = ((Xtr - MEAN) / STD).astype(np.float32)
Tva = ((Xva - MEAN) / STD).astype(np.float32)
print(D, Ttr.shape, Tva.shape)

In [ ]:
# 3b. Cài thư viện
import subprocess, sys, importlib, warnings
warnings.filterwarnings("ignore")

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

print("mmengine + mmcv-lite:", pip("install", "-q", "mmengine", "mmcv-lite==2.1.0")[0])
print("mmaction2:", pip("install", "-q", "--no-deps", "mmaction2")[0])
importlib.invalidate_caches()
import mmcv, mmengine, mmaction
try: from mmaction.models.backbones import STGCN as MMActionSTGCN
except ModuleNotFoundError: from mmaction.models.backbones import STGCN as MMActionSTGCN
print("mmcv", mmcv.__version__, "| mmengine", mmengine.__version__, "| mmaction2", mmaction.__version__)

In [ ]:
# 4. Mô hình ST-GCN từ MMAction2
HAND_E = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
          (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]
EDGES = HAND_E + [(a + 21, b + 21) for a, b in HAND_E] + [(42, 43), (42, 46), (43, 46), (43, 44), (44, 45), (46, 47), (47, 48), (45, 0), (48, 21)]
V = 49
XN = np.r_[np.arange(0, 84, 2), 98, 84, 86, 88, 90, 92, 94]
GRAPH = dict(layout=dict(num_node=V, inward=[tuple(e) for e in EDGES], center=42), mode="spatial")

class STGCN(nn.Module):
    def __init__(self, n_cls=N_CLS, drop=HEAD_DROPOUT):
        super().__init__()
        self.register_buffer("xi", torch.from_numpy(XN).long())
        self.register_buffer("yi", torch.from_numpy(XN + 1).long())
        self.enc = MMActionSTGCN(GRAPH, in_channels=8, base_channels=STGCN_WIDTH, num_person=1)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(STGCN_WIDTH * 4, n_cls)

    def embed(self, x):
        B, T = x.shape[:2]
        xy = torch.stack([x[..., self.xi], x[..., self.yi]], -1)
        d1 = xy - torch.cat([xy[:, :1], xy[:, :-1]], 1)
        d2 = xy - torch.cat([xy[:, :1], xy[:, :1], xy[:, :-2]], 1)
        one, zero = torch.ones(B, T, 7, device=x.device), torch.zeros(B, T, 7, device=x.device)
        flag = torch.cat([x[..., 100:101].expand(B, T, 21), x[..., 101:102].expand(B, T, 21), one], -1)
        size = torch.cat([x[..., 96:97].expand(B, T, 21), x[..., 97:98].expand(B, T, 21), zero], -1)
        h = torch.cat([xy, flag[..., None], size[..., None], d1, d2], -1)
        return self.enc(h[:, None]).mean((1, 3, 4))

    def forward(self, x):
        return self.fc(self.drop(self.embed(x)))

BUILD = {"stgcn": STGCN}
net = STGCN()
print(f"stgcn {sum(p.numel() for p in net.parameters()):>11,} tham số, đầu ra {tuple(net(torch.randn(2, 64, D)).shape)}")

In [ ]:
# 5. Hàm đánh giá
def confusion(y, p):
    cm = np.zeros((N_CLS, N_CLS), int)
    np.add.at(cm, (y, p), 1)
    return cm

def f1_per_class(y, p):
    cm = confusion(y, p)
    tp = np.diag(cm).astype(float)
    prec = tp / np.maximum(cm.sum(0), 1)
    rec = tp / np.maximum(cm.sum(1), 1)
    return np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0.0)

def topk(y, prob, k):
    return float((np.argsort(-prob, 1)[:, :k] == y[:, None]).any(1).mean())

def metrics(y, prob):
    p = prob.argmax(1)
    present = np.unique(y)
    return dict(acc=float((p == y).mean()), f1=float(f1_per_class(y, p)[present].mean()), top3=topk(y, prob, 3), top5=topk(y, prob, 5))

def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def ece(y, prob, bins=10):
    conf, pred = prob.max(1), prob.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for a, b in zip(edges[:-1], edges[1:]):
        m = (conf > a) & (conf <= b)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

In [ ]:
# 6. Vòng train
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict(model, X, bs=64):
    model.eval()
    return np.concatenate([F.softmax(model(torch.from_numpy(X[i:i + bs]).to(DEV)), -1).cpu().numpy() for i in range(0, len(X), bs)])

def fit(name, seed):
    set_seed(seed)
    rng = np.random.default_rng(seed)
    run_dir = f"{OUT}/runs/{name}_s{seed}"
    os.makedirs(run_dir, exist_ok=True)
    model = BUILD[name]().to(DEV)
    n_par = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    hist, best, best_ep, best_state, wait, t0 = [], -1.0, 0, None, 0, time.time()
    for ep in range(1, EPOCHS + 1):
        model.train()
        te = time.time()
        perm = rng.permutation(len(Ttr))
        tl, tc = 0.0, 0
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            xb, yb = torch.from_numpy(Ttr[b]).to(DEV), torch.from_numpy(ytr[b]).to(DEV)
            out = model(xb)
            loss = F.cross_entropy(out, yb, label_smoothing=LABEL_SMOOTH)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()
            tl += loss.item() * len(b)
            tc += (out.argmax(1) == yb).sum().item()
        pv = predict(model, Tva)
        vl = float(-np.log(pv[np.arange(len(yva)), yva] + 1e-9).mean())
        mv = metrics(yva, pv)
        hist.append(dict(epoch=ep, train_loss=tl / len(perm), train_acc=tc / len(perm), val_loss=vl,
                         val_acc=mv["acc"], val_f1=mv["f1"], epoch_s=time.time() - te))
        if mv["f1"] > best:
            best, best_ep, wait = mv["f1"], ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            np.save(f"{run_dir}/val_probs.npy", pv)
        else:
            wait += 1
        if ep % 10 == 0 or ep == 1:
            print(f"{name} seed {seed} epoch {ep:3d}  loss {tl / len(perm):.3f}/{vl:.3f}  acc {tc / len(perm):.3f}/{mv['acc']:.3f}  F1 {mv['f1']:.3f}  best {best:.3f}@{best_ep}")
        if wait >= PATIENCE:
            print(f"{name} seed {seed} dừng sớm ở epoch {ep} (tốt nhất epoch {best_ep})")
            break
    pd.DataFrame(hist).to_csv(f"{run_dir}/history.csv", index=False)
    model.load_state_dict(best_state)
    torch.save(dict(model=name, state_dict=best_state, epoch=best_ep, seed=seed, mean=MEAN, std=STD, names=NAMES), f"{run_dir}/best.pt")
    mv = metrics(yva, np.load(f"{run_dir}/val_probs.npy"))
    tr_acc = metrics(ytr, predict(model, Ttr))["acc"]
    return dict(model=name, seed=seed, params=n_par, best_epoch=best_ep, stop_epoch=ep, val_acc=mv["acc"], val_f1=mv["f1"], val_top3=mv["top3"],
                val_top5=mv["top5"], val_loss=hist[best_ep - 1]["val_loss"], train_acc=tr_acc,
                epoch_s=float(np.mean([x["epoch_s"] for x in hist])), time_s=round(time.time() - t0, 1))

In [ ]:
# 7. Train ST-GCN × seed, early stopping
ROWS = [fit("stgcn", s) for s in SEEDS]
pd.DataFrame(ROWS).to_csv(f"{OUT}/summary.csv", index=False)

In [ ]:
# 8. Kết quả val
res = pd.DataFrame(ROWS)
display(res.round(4))
PRB = np.stack([np.load(f"{OUT}/runs/stgcn_s{s}/val_probs.npy") for s in SEEDS])
ENS = PRB.mean(0)
cols = ["val_acc", "val_f1", "val_top3", "val_top5", "val_loss", "best_epoch", "stop_epoch"]
summ = pd.DataFrame({"mean": res[cols].mean(), "std": res[cols].std()})
em = metrics(yva, ENS)
summ["ensemble"] = [em["acc"], em["f1"], em["top3"], em["top5"], np.nan, np.nan, np.nan]
display(summ.round(4))
summ.round(4).to_csv(f"{OUT}/val_summary.csv")
lo_, hi_ = wilson(int(round(em["acc"] * len(yva))), len(yva))
print(f"ensemble acc {em['acc']:.3f} (95%: {lo_:.3f}–{hi_:.3f}) | macro-F1 {em['f1']:.3f} | ECE {ece(yva, ENS):.3f} | ECE trung bình seed {np.mean([ece(yva, p) for p in PRB]):.3f}")

In [ ]:
# 9. Đường học
H = [pd.read_csv(f"{OUT}/runs/stgcn_s{s}/history.csv").set_index("epoch") for s in SEEDS]

def band(a, col, ls="-", label=None):
    d = pd.concat([h[col] for h in H], axis=1)
    mu, sd = d.mean(1), d.std(1).fillna(0)
    a.plot(mu.index, mu, ls, c="C0", label=label or col)
    if ls == "-":
        a.fill_between(mu.index, mu - sd, mu + sd, color="C0", alpha=.15)

fig, ax = plt.subplots(3, 1, figsize=(11, 15))
band(ax[0], "val_loss", label="val"); band(ax[0], "train_loss", "--", "train")
band(ax[1], "val_acc", label="val"); band(ax[1], "train_acc", "--", "train")
band(ax[2], "val_f1", label="val macro-F1")
ax[0].set_title("loss"); ax[1].set_title("accuracy"); ax[2].set_title("val macro-F1")
for a in ax:
    a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/learning_curve.png", dpi=110); plt.show()

In [ ]:
# 10. Confusion matrix val
CM = confusion(yva, ENS.argmax(1))
cmn = CM / np.maximum(CM.sum(1, keepdims=True), 1)
fig, a = plt.subplots(figsize=(11, 10.5))
a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
a.set_xticks(range(N_CLS)); a.set_xticklabels(NAMES, rotation=90, fontsize=7)
a.set_yticks(range(N_CLS)); a.set_yticklabels(NAMES, fontsize=7)
for i, j in zip(*np.nonzero(CM)):
    a.text(j, i, CM[i, j], ha="center", va="center", fontsize=6, color="w" if cmn[i, j] > .5 else "k")
a.set_title(f"ST-GCN ensemble {len(SEEDS)} seed, val"); a.set_xlabel("dự đoán"); a.set_ylabel("thật")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_val.png", dpi=110); plt.show()

In [ ]:
# 11. F1 theo lớp và cặp nhầm
F1C = f1_per_class(yva, ENS.argmax(1))
fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(range(N_CLS), F1C, color="C0")
ax.set_xticks(range(N_CLS)); ax.set_xticklabels(NAMES, rotation=60, fontsize=8); ax.set_title("F1 theo từng lớp (ensemble, val)")
plt.tight_layout(); plt.savefig(f"{OUT}/f1_per_class.png", dpi=110); plt.show()
pd.DataFrame({"tu": NAMES, "f1": F1C.round(4)}).to_csv(f"{OUT}/f1_per_class.csv", index=False)
off = CM.copy()
np.fill_diagonal(off, 0)
idx = np.dstack(np.unravel_index(np.argsort(-off, axis=None)[:10], off.shape))[0]
pairs = pd.DataFrame([(NAMES[i], NAMES[j], int(off[i, j])) for i, j in idx if off[i, j]], columns=["thật", "đoán nhầm thành", "số lần"])
pairs.to_csv(f"{OUT}/confusion_pairs.csv", index=False)
display(pairs)

In [ ]:
# 12. OK và LOW_HAND
rows = []
for s in ("OK", "LOW_HAND"):
    k = sva == s
    em_k = metrics(yva[k], ENS[k])
    accs = [metrics(yva[k], p[k])["acc"] for p in PRB]
    lo_, hi_ = wilson(int(round(em_k["acc"] * k.sum())), int(k.sum()))
    rows.append(dict(status=s, n=int(k.sum()), acc_ensemble=em_k["acc"], f1_ensemble=em_k["f1"], acc_seed_mean=np.mean(accs),
                     acc_seed_std=np.std(accs, ddof=1) if len(accs) > 1 else np.nan, ci95_lo=lo_, ci95_hi=hi_))
st = pd.DataFrame(rows).set_index("status")
st.round(4).to_csv(f"{OUT}/ok_vs_low_hand.csv")
display(st.round(3))

In [ ]:
# 13. Mẫu sai val
err = pd.DataFrame(dict(sample_id=iva, participant_id=gva, status=sva, that=np.array(NAMES)[yva], doan=np.array(NAMES)[ENS.argmax(1)],
                        p=ENS.max(1).round(3), top3=[", ".join(NAMES[j] for j in np.argsort(-r)[:3]) for r in ENS]))
err = err[err.that != err.doan]
err.to_csv(f"{OUT}/val_errors.csv", index=False)
print(f"{len(err)} mẫu sai / {len(yva)}")
display(err.head(15))

In [ ]:
# 14. 5 mẫu đúng và 5 mẫu sai
P = ENS
pred = P.argmax(1)
rng = np.random.default_rng(0)

def pick(mask, k=5):
    out, seen = [], set()
    for i in rng.permutation(np.flatnonzero(mask)):
        if yva[i] not in seen:
            out.append(i)
            seen.add(yva[i])
        if len(out) == k:
            break
    return out

rows = [("ĐÚNG", i) for i in pick(pred == yva)] + [("SAI", i) for i in pick(pred != yva)]
fig, ax = plt.subplots(len(rows), 4, figsize=(12, 3.2 * len(rows)))
fig.suptitle("Cột 1: quỹ đạo cổ tay (xanh = phải, cam = trái; tròn = khung đầu, vuông = khung cuối). Cột 2–4: khung xương bàn tay tại 3 khung", fontsize=9, y=1.002)
for r, (tag, i) in enumerate(rows):
    x = Xva[i]
    a = ax[r, 0]
    for c, col in ((88, "C0"), (92, "C1")):
        a.plot(x[:, c], x[:, c + 1], "-", c=col, alpha=.6)
        a.plot(x[0, c], x[0, c + 1], "o", c=col)
        a.plot(x[-1, c], x[-1, c + 1], "s", c=col)
    a.invert_yaxis(); a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
    a.set_title(f"{tag}: thật {NAMES[yva[i]]}, đoán {NAMES[pred[i]]} (p={P[i].max():.2f}, {sva[i]})", fontsize=8, loc="left")
    hand = 0 if x[:, 100].sum() >= x[:, 101].sum() else 1
    vis = np.flatnonzero(x[:, 100 + hand] > .5)
    if len(vis) == 0:
        vis = np.arange(len(x))
    for k, f in enumerate(vis[np.linspace(0, len(vis) - 1, 3).astype(int)]):
        b = ax[r, 1 + k]
        pts = x[f, 42 * hand:42 * hand + 42].reshape(21, 2)
        for u, v in HAND_E:
            b.plot(pts[[u, v], 0], pts[[u, v], 1], "k-", lw=1)
        b.plot(pts[:, 0], pts[:, 1], "r.", ms=4)
        b.invert_yaxis(); b.set_aspect("equal"); b.set_xticks([]); b.set_yticks([])
        b.set_title(f"khung {f}, tay {'phải' if hand == 0 else 'trái'}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/mau_dung_sai.png", dpi=100, bbox_inches="tight"); plt.show()

In [ ]:
# 15. Xuất mô hình cho demo
import mmaction
EXP = f"{OUT}/export"
os.makedirs(EXP, exist_ok=True)

class Export(nn.Module):
    def __init__(self, nets):
        super().__init__()
        self.nets = nn.ModuleList(nets)
        self.register_buffer("mean", torch.from_numpy(MEAN))
        self.register_buffer("std", torch.from_numpy(STD))

    def forward(self, x):
        z = (x - self.mean) / self.std
        return torch.stack([F.softmax(n(z), -1) for n in self.nets]).mean(0)

def load_cpu(s):
    net = STGCN()
    net.load_state_dict(torch.load(f"{OUT}/runs/stgcn_s{s}/best.pt", map_location="cpu", weights_only=False)["state_dict"])
    return net.eval()

NETS = {s: load_cpu(s) for s in SEEDS}
BEST_S = int(res.seed[res.val_f1.idxmax()])
VARIANTS = {"ensemble": Export(list(NETS.values())).eval(), "single": Export([NETS[BEST_S]]).eval()}
torch.save(dict(state_dicts={s: n.state_dict() for s, n in NETS.items()}, best_seed=BEST_S, mean=MEAN, std=STD, names=NAMES,
                stgcn_width=STGCN_WIDTH, edges=EDGES, node_x_index=XN.tolist()), f"{EXP}/stgcn_asl20_state.pt")
FILES = {}
for k, mod in VARIANTS.items():
    traced = torch.jit.trace(mod, torch.randn(2, 64, D))
    FILES[k] = f"{EXP}/stgcn_asl20_{k}.pt"
    traced.save(FILES[k])
    chk = torch.jit.load(FILES[k])
    for b in (1, 3):
        xb = torch.randn(b, 64, D)
        with torch.no_grad():
            assert (chk(xb) - mod(xb)).abs().max().item() < 1e-4, (k, b)
with torch.no_grad():
    pj = torch.jit.load(FILES["ensemble"])(torch.from_numpy(Xva[:8])).numpy()
assert np.abs(pj - ENS[:8]).max() < 1e-3, np.abs(pj - ENS[:8]).max()
print("TorchScript khớp mô hình gốc; dự đoán 8 mẫu val đầu:", [NAMES[i] for i in pj.argmax(1)], "| thật:", [NAMES[i] for i in yva[:8]])
META = dict(
    task="ASL 20-sign classification from MediaPipe Holistic keypoints, 64 frames per clip",
    classes=NAMES, num_frames=64, feature_dim=int(D),
    input="float32 tensor (batch, 64, 102): raw features as produced by the data-prep notebook, NOT standardized (mean/std are inside the model)",
    output="float32 tensor (batch, 20): softmax probabilities, same order as `classes`",
    feature_layout={"0-41": "right hand 21 landmarks x,y (wrist-relative, hand-size normalized)", "42-83": "left hand 21 landmarks x,y",
                    "84-95": "right shoulder, elbow, wrist then left shoulder, elbow, wrist, x,y (shoulder-relative)",
                    "96-97": "hand size right, left", "98-99": "nose x,y", "100-101": "hand-visible flag right, left (0/1)"},
    files={"stgcn_asl20_ensemble.pt": f"TorchScript, mean of {len(SEEDS)} seeds (more accurate, slower)",
           "stgcn_asl20_single.pt": f"TorchScript, seed {BEST_S} only (faster)",
           "stgcn_asl20_state.pt": "state_dict of every seed; loading needs mmaction2 and the STGCN wrapper of this notebook"},
    usage="m = torch.jit.load('stgcn_asl20_ensemble.pt'); probs = m(x)  # x: (B, 64, 102) float32; class = classes[probs.argmax(1)]",
    model=f"ST-GCN (MMAction2, base_channels={STGCN_WIDTH}), trained from scratch", params=int(res.params.iloc[0]), seeds=SEEDS, best_seed=BEST_S,
    val=dict(f1_seed_mean=float(res.val_f1.mean()), f1_ensemble=float(em["f1"]), acc_ensemble=float(em["acc"])),
    torch=torch.__version__, mmaction2=mmaction.__version__)
json.dump(META, open(f"{EXP}/meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
for f in sorted(os.listdir(EXP)):
    print(f"{f:28s} {os.path.getsize(f'{EXP}/{f}') / 1e6:8.1f} MB")

In [ ]:
# 16. Test, chỉ chạy một lần
if RUN_FINAL_TEST:
    lock = f"{OUT}/TEST_DONE.lock"
    assert not os.path.exists(lock), "test đã chạy rồi"
    open(lock, "w").write(time.ctime())
    Xte, yte, gte_, ite = load(EVAL_SPLIT)
    Tte = ((Xte - MEAN) / STD).astype(np.float32)
    ste = meta.drop_duplicates("sample_id").set_index("sample_id").status.reindex(ite).to_numpy()
    TP = []
    for s in SEEDS:
        mdl = STGCN().to(DEV)
        mdl.load_state_dict(torch.load(f"{OUT}/runs/stgcn_s{s}/best.pt", map_location=DEV, weights_only=False)["state_dict"])
        TP.append(predict(mdl, Tte))
    TP = np.stack(TP)
    TE = TP.mean(0)
    per_seed = pd.DataFrame([metrics(yte, p) for p in TP])
    tres = pd.DataFrame({"mean": per_seed.mean(), "std": per_seed.std(), "ensemble": pd.Series(metrics(yte, TE))}).round(4)
    tres.to_csv(f"{OUT}/test_metrics.csv")
    print(f"{EVAL_SPLIT.upper()} ({len(yte)} mẫu, {len(set(gte_))} người ký)")
    display(tres)
    for s_ in ("OK", "LOW_HAND"):
        k = ste == s_
        if k.any():
            print(f"{s_:9s} {int(k.sum()):3d} mẫu | acc ensemble {metrics(yte[k], TE[k])['acc']:.4f}")
    CMT = confusion(yte, TE.argmax(1))
    cmn = CMT / np.maximum(CMT.sum(1, keepdims=True), 1)
    fig, a = plt.subplots(figsize=(11, 10.5))
    a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    a.set_xticks(range(N_CLS)); a.set_xticklabels(NAMES, rotation=90, fontsize=7)
    a.set_yticks(range(N_CLS)); a.set_yticklabels(NAMES, fontsize=7)
    for i, j in zip(*np.nonzero(CMT)):
        a.text(j, i, CMT[i, j], ha="center", va="center", fontsize=6, color="w" if cmn[i, j] > .5 else "k")
    a.set_title(f"ST-GCN ensemble {len(SEEDS)} seed, {EVAL_SPLIT}"); a.set_xlabel("dự đoán"); a.set_ylabel("thật")
    plt.tight_layout(); plt.savefig(f"{OUT}/confusion_{EVAL_SPLIT}.png", dpi=110); plt.show()
    terr = pd.DataFrame(dict(sample_id=ite, participant_id=gte_, status=ste, that=np.array(NAMES)[yte], doan=np.array(NAMES)[TE.argmax(1)],
                             p=TE.max(1).round(3), top3=[", ".join(NAMES[j] for j in np.argsort(-r)[:3]) for r in TE]))
    terr = terr[terr.that != terr.doan]
    terr.to_csv(f"{OUT}/{EVAL_SPLIT}_errors.csv", index=False)
    print(f"{len(terr)} mẫu sai / {len(yte)}")
    display(terr)
    META[EVAL_SPLIT] = dict(f1_seed_mean=float(per_seed.f1.mean()), f1_ensemble=float(tres.loc["f1", "ensemble"]), acc_ensemble=float(tres.loc["acc", "ensemble"]), n=int(len(yte)))
    json.dump(META, open(f"{EXP}/meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
else:
    print("RUN_FINAL_TEST=False, chưa đụng tới test")